# ISMIP7 forcing on StratusGeo (CryoCloud)

The ISMIP7 forcing datasets on [source.coop](https://source.coop/ismip) are about
285,000 NetCDF files (6 TB). The [englacial/ismip7](https://source.coop/englacial/ismip7)
Icechunk stores hold *virtual references* into those files (22 GB of metadata, no data
copied). One store per ice sheet, `icechunk-ais` and `icechunk-gris`, laid out as

    <ESM>/<experiment>/<product>/<resolution>m      e.g. CESM2-WACCM/ssp585/SDBN1/2000m

Each resolution group is one xarray Dataset: a normalised monthly (or annual) time axis
in `days since 1850-01-01`, the exact ISMIP7 grid in `x`/`y`, and a `crs` variable with
the EPSG code and WKT. The group above the resolutions (the *family*, e.g.
`CESM2-WACCM/ssp585/SDBN1`) carries an OME-NGFF `multiscales` attribute naming its
levels, so a multiscale-aware reader (gridlook) opens the family as one pyramid.

Reading needs no credentials: `icechunk` opens the store, `xarray` reads it. The
hierarchy is walked with [rustytree](https://github.com/aladinor/rustytree) when it is
installed (a concurrent walk in Rust: the whole store in seconds), falling back to
zarr-python, which does the same one group at a time.


## Install what the image lacks

In [ ]:
# the StratusGeo (CryoCloud) image has zarr, xarray and hvplot but not icechunk or
# rustytree; this installs them into your home directory, which persists, so it is
# needed once per hub. Wheels only: a source build of rustytree needs a Rust toolchain
# the hub lacks and would run for a very long time before failing.
%pip install icechunk
%pip install --only-binary :all: rustytree-xarray

## Open the store

In [ ]:
import importlib.util

import icechunk
import xarray as xr

# the Antarctic store (Greenland: prefix="englacial/ismip7/icechunk-gris")
storage = icechunk.s3_storage(
    bucket="us-west-2.opendata.source.coop",
    prefix="englacial/ismip7/icechunk-ais",
    region="us-west-2",
    anonymous=True,
)
# the store holds references into the ISMIP7 NetCDF files; icechunk follows them
# only into a bucket you have authorised
containers = icechunk.containers_credentials(
    {"s3://us-west-2.opendata.source.coop/ismip/": icechunk.s3_anonymous_credentials()}
)
repo = icechunk.Repository.open(storage, authorize_virtual_chunk_access=containers)
store = repo.readonly_session("main").store

# the series run to 2300, past datetime64[ns]: decode time with cftime
TIME = xr.coders.CFDatetimeCoder(use_cftime=True)
# rustytree opens the whole store in seconds (concurrent walk in Rust); zarr works too
ENGINE = "rustytree" if importlib.util.find_spec("rustytree") else "zarr"

## The whole store is one lazy tree

In [ ]:
%%time
tree = xr.open_datatree(store, engine=ENGINE, consolidated=False, decode_times=TIME)
list(tree.children)

In [ ]:
# groups are ESM / experiment / family / resolution
family = tree["CESM2-WACCM/historical/SDBN1"]
family

In [ ]:
# the resolutions of a family, finest first (OME-NGFF multiscales)
family.attrs["multiscales"][0]["datasets"]

## A resolution is a Dataset, a variable is a DataArray

In [ ]:
# one resolution is a Dataset
ds8 = family["8000m"].to_dataset()
ds2 = family["2000m"].to_dataset()
ds8

In [ ]:
acabf = ds8["acabf"]
print(acabf.attrs["long_name"], acabf.attrs["units"], "| colour range", acabf.attrs["actual_range"])
print(acabf.time.values[0], "->", acabf.time.values[-1], f"({acabf.sizes['time']} steps)")

## Two resolutions of the same field

In [ ]:
# the same month at 8 km and 2 km, panes linked: zoom in and the 2 km detail appears
# (rasterize draws at screen resolution and redraws on every zoom)
# under a JupyterHub (Binder, CryoCloud) Panel asks for its JavaScript at the wrong path
# and shows "could not load resources from its Jupyter server extension"; the CDN avoids it
%env PANEL_NOTEBOOK_RESOURCES=cdn
import holoviews as hv
import hvplot.xarray  # noqa: F401

month8 = ds8["acabf"].isel(time=1900)
month2 = ds2["acabf"].isel(time=1900)
date = str(month8.time.values)[:10]
pane8 = month8.hvplot.image(x="x", y="y", rasterize=True, title=f"8 km, {date}")
pane2 = month2.hvplot.image(x="x", y="y", rasterize=True, title=f"2 km, {date}")
(pane8 + pane2).opts(hv.opts.Image(cmap="viridis", frame_width=420, frame_height=420))

In [ ]:
# a 400 km window at 2 km, averaged over the first year; nothing is read until the plot asks
box = ds2["acabf"].isel(time=slice(0, 12), x=slice(1400, 1600), y=slice(1400, 1600))
box.mean("time").hvplot.image(x="x", y="y", cmap="viridis", frame_width=400, frame_height=400)

## Everything in the store

In [ ]:
# every family in the store, with its resolutions
for node in tree.subtree:
    if "multiscales" in node.attrs:
        print(node.path, [d["path"] for d in node.attrs["multiscales"][0]["datasets"]])

## Greenland

In [ ]:
%%time
# Greenland: a second store with the same layout (EPSG:3413 at 4 km and 1 km)
storage_gris = icechunk.s3_storage(
    bucket="us-west-2.opendata.source.coop",
    prefix="englacial/ismip7/icechunk-gris",
    region="us-west-2",
    anonymous=True,
)
repo_gris = icechunk.Repository.open(storage_gris, authorize_virtual_chunk_access=containers)
gris = xr.open_datatree(
    repo_gris.readonly_session("main").store, engine=ENGINE, consolidated=False, decode_times=TIME
)
gris["CESM2-WACCM/historical/SDBN1"]